# Fig. 2 — `fig:activity` (`paper/figs/fig_forced.pdf`)

Tách từ `evaluation/paper_figures.py::fig_forced`. Chạy từ thư mục `notebooks/`, kernel `.venv`; cần log trong `output_logs/macsql_drspider/{openai-gpt-5.6-luna,ollama-qwen2.5-14b}[_selector-force]/`.

Cell 1: setup (palette, loader log, Wilson CI). Cell 2: vẽ hình, ghi `paper/figs/fig_forced.pdf`.

In [ ]:
"""
evaluation/paper_figures.py

Every figure in paper/soict2026.tex, regenerated from the logs so that no
figure is a file whose provenance has been forgotten. Writes vector PDFs to
paper/figs/.

    python evaluation/paper_figures.py            # all
    python evaluation/paper_figures.py activity   # one, by name

Palette: Okabe–Ito (colour-blind safe), assigned by entity in fixed order and
never cycled. Backbone identity: luna = blue, Qwen = orange. Stage activity:
inert = grey, active = blue. One axis per panel; no dual axes.
"""
import glob
import json
import math
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path.cwd().resolve().parent  # notebook lives in <repo>/notebooks/
sys.path.insert(0, str(ROOT / "evaluation"))
from cross_system_report import GT_STAGE, attribute_baseline, columns_kept, gold_column_refs  # noqa: E402
from label_validity import classify, VERDICTS  # noqa: E402

OUT = ROOT / "paper" / "figs"
OUT.mkdir(exist_ok=True)

STAGES = ["Intent", "Schema", "Skeleton", "Execution"]
BACKBONES = [("gpt-5.6-luna", "openai-gpt-5.6-luna"),
             ("Qwen2.5-14B", "ollama-qwen2.5-14b")]

# Okabe–Ito
C_BLUE, C_ORANGE, C_GREEN, C_VERM, C_GREY, C_SKY, C_PURPLE = (
    "#0072B2", "#E69F00", "#009E73", "#D55E00", "#8C8C8C", "#56B4E9", "#CC79A7")
C_BB = {"gpt-5.6-luna": C_BLUE, "Qwen2.5-14B": C_ORANGE}
C_ACT = {"inert": C_GREY, "pruned": C_BLUE}
INK, INK2 = "#222222", "#555555"

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.linewidth": 0.6,
    "axes.edgecolor": INK2,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.color": INK2,
    "ytick.color": INK2,
    "axes.labelcolor": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "legend.frameon": False,
    "legend.fontsize": 7,
})


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _read_json_objects(path: Path) -> list:
    """Some counterfactual files hold several concatenated JSON objects."""
    txt = path.read_text(encoding="utf-8")
    dec = json.JSONDecoder()
    i, out = 0, []
    while i < len(txt):
        while i < len(txt) and txt[i] in " \n\r\t":
            i += 1
        if i >= len(txt):
            break
        try:
            d, i = dec.raw_decode(txt, i)
        except json.JSONDecodeError:
            break
        out.append(d)
    return out


def load_drspider(backbone_dir: str) -> list:
    path = ROOT / "output_logs" / "macsql_drspider" / backbone_dir
    out = []
    for f in sorted(glob.glob(str(path / "drspider_*.json"))):
        try:
            out.append(json.loads(Path(f).read_text(encoding="utf-8")))
        except Exception:
            continue
    return out


def load_bird(system: str) -> list:
    path = ROOT / "output_logs" / f"{system}_bird" / "openai-gpt-5.6-luna"
    out = []
    for f in sorted(glob.glob(str(path / "bird_*.json"))):
        try:
            out.append(json.loads(Path(f).read_text(encoding="utf-8")))
        except Exception:
            continue
    return out


def load_counterfactual(arm: str) -> dict:
    path = ROOT / "output_logs" / "counterfactual" / "openai-gpt-5.6-luna" / arm
    m = {}
    for f in glob.glob(str(path / "*.json")):
        for d in _read_json_objects(Path(f)):
            if isinstance(d, dict) and "benchmark_id" in d:
                m[d["benchmark_id"]] = d
    return m


def pruned(d: dict) -> bool:
    return bool((d.get("x5") or {}).get("pruned"))


def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#DDDDDD", lw=0.5, zorder=0)
    ax.set_axisbelow(True)




In [ ]:
def fig_forced() -> None:
    """DB group under the default sweep and the forced-Selector arm.

    Four horizontal bars per backbone: instances the default Selector left inert vs
    pruned, each under the default sweep and under the arm that forces the
    Selector to act. (a) EX barely moves; (b) schema recall on failures is
    zero on inert instances only while the Selector is inert. A backbone
    without a forced arm on disk is drawn with its default bars only.
    """
    # xcolor mixes from the tikz pipeline figure.  Hue = Selector activity in
    # the default sweep (inert = orange, pruned = blue); shade = arm (default =
    # light tint, forced = dark tint of the same hue).  No hatching.
    C_FILL = {("inert", "default"): "#FFE3C7",   # orange!22
              ("inert", "forced"): "#BF6000",    # orange!75!black
              ("pruned", "default"): "#D1D1FF",  # blue!18
              ("pruned", "forced"): "#000099"}   # blue!60!black
    C_EDGE = {"inert": "#BF6000", "pruned": "#000099"}
    rows = {}
    for bb, bdir in BACKBONES:
        base = {d["benchmark_id"]: d for d in load_drspider(bdir)
                if d.get("perturbation_group") == "DB"}
        arm = {d["benchmark_id"]: d for d in load_drspider(bdir + "_selector-force")
               if d.get("perturbation_group") == "DB"}
        for act in ("inert", "pruned"):
            ids = [i for i in base if pruned(base[i]) == (act == "pruned")]
            for arm_name, logs in (("default", base), ("forced", arm)):
                sub = [logs[i] for i in ids if i in logs]
                if len(sub) < len(ids):  # arm not run for this backbone
                    continue
                fails = [d for d in sub if not d.get("execution_match")]
                rows[(bb, act, arm_name)] = dict(
                    n=len(sub), ex=sum(bool(d.get("execution_match")) for d in sub),
                    nf=len(fails), hit=sum(attribute_baseline(d) == "Schema" for d in fails))

    fig, axes = plt.subplots(1, 2, figsize=(4.8, 2.7), sharey=True)
    h = 0.19
    order = [("inert", "default"), ("inert", "forced"), ("pruned", "default"), ("pruned", "forced")]
    for ax, key, xlabel, title in (
        (axes[0], "ex", "execution accuracy (%)", "(a) EX unchanged when forced"),
        (axes[1], "rec", "schema recall on failures (%)", "(b) recall zero only while inert"),
    ):
        for j, (bb, _) in enumerate(BACKBONES):
            for k, (act, arm_name) in enumerate(order):
                r = rows.get((bb, act, arm_name))
                y = j + (k - 1.5) * (h + 0.03)
                if r is None:
                    continue
                num, den = (r["ex"], r["n"]) if key == "ex" else (r["hit"], r["nf"])
                v = 100 * num / den if den else 0
                lo, hi = wilson(num, den)
                ax.barh(y, v, h, facecolor=C_FILL[(act, arm_name)], edgecolor=C_EDGE[act],
                        lw=0.7, zorder=3)
                ax.errorbar(v, y, xerr=[[v - 100 * lo], [100 * hi - v]],
                            fmt="none", ecolor=INK, elinewidth=0.7, capsize=1.5, zorder=4)
                ax.text(100 * hi + 1.5, y, f"{num}/{den}", ha="left", va="center",
                        fontsize=6.5, color=INK)
        ax.set_yticks([0, 1])
        ax.set_yticklabels([bb for bb, _ in BACKBONES], fontsize=8)
        ax.tick_params(axis="x", labelsize=7)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_xlim(0, 128 if key == "ex" else 100)
        ax.set_xticks(range(0, 101 if key == "ex" else 81, 20))
        ax.set_title(title, fontsize=7.5, loc="left")
        _style(ax)
        ax.grid(axis="x", color="#D9D9D9", lw=0.5, zorder=0)  # black!15
        ax.grid(axis="y", visible=False)
    axes[0].invert_yaxis()  # shared y: first backbone on top, bars read top-to-bottom in `order`
    fig.legend(handles=[Patch(facecolor=C_FILL[(act, arm)], edgecolor=C_EDGE[act], lw=0.7,
                              label=f"Selector {act} in default sweep, {'forced to act' if arm == 'forced' else 'default'}")
                        for act in ("inert", "pruned") for arm in ("default", "forced")],
               ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.15), handlelength=1.4,
               fontsize=7, columnspacing=1.2)
    fig.tight_layout(w_pad=1.5)
    fig.savefig(OUT / "fig_forced.pdf", bbox_inches="tight")
    print("wrote", OUT / "fig_forced.pdf")
    return fig


fig = fig_forced()
